# 113 — Q-gradient PCP proposal diagnostic (offline only)

At held-out v4 root states, compute a bounded gradient step on the stored P&P action chunk and measure its Q-score change and normalized action displacement. **No corrected action has an observed outcome.** A higher predicted Q is not evidence of better control. The next stage is a matched simulator evaluation of these new chunks, with a stock/P&P fallback and a fixed trust radius. This notebook does not claim to implement or evaluate the old `pcp.py` MLP corrector.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXTRAS = 'sim'
SETUP_ENV = False
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
import numpy as np
import torch
from pnp.smolvla_q_selection import (
    FRESH8_EXPERIMENT, FRESH8_KINDS, fresh8_snapshot_key,
    load_success_q10, bounded_q_ascent)
from pnp.smolvla_success_critic import TimedRoots
from pnp.smolvla_tree_bellman_finetune import load_or_create_tree_snapshot, prepare_tree_bellman_cache
from pnp.store import SupabaseStore

CHECKPOINT = Path('REPLACE_WITH_NOTEBOOK_109_PREFILL_Q_CHECKPOINT.pt')
TREE_LIMIT = 800
MAX_ROOTS = 16
TRUST_RADIUS_STD = 0.05
RUN_DIAGNOSTIC = False
assert CHECKPOINT.is_file(), 'Set CHECKPOINT to a saved prefill-prefix Q10 checkpoint.'
store = SupabaseStore()
snapshot = load_or_create_tree_snapshot(
    store=store, snapshot_key=fresh8_snapshot_key(TREE_LIMIT), tree_limit=TREE_LIMIT,
    experiment=FRESH8_EXPERIMENT, candidate_kinds=FRESH8_KINDS)
cache = prepare_tree_bellman_cache(
    snapshot=snapshot, cache_root=Path('/content/smolvla_q10_fresh8_cache'),
    gamma=1.0, success_reward=True, store=store)
model, payload = load_success_q10(CHECKPOINT,
    expected_snapshot=snapshot['snapshot_digest'], device='cuda')
assert payload.get('representation', 'frozen_prefill_prefix_v1') == 'frozen_prefill_prefix_v1'
groups = {g['candidate_group_id']: g for g in snapshot['groups']}
roots = TimedRoots(cache, cache['validation_group_ids'], groups)


In [ ]:
if RUN_DIAGNOSTIC:
    records = []
    for i in range(min(MAX_ROOTS, len(roots))):
        row = roots[i]
        prefix = torch.as_tensor(row['prefix'][None], device='cuda')
        pad = torch.as_tensor(row['pad'][None], device='cuda')
        robot = torch.as_tensor(row['robot'][None], device='cuda')
        proprio = torch.as_tensor(row['proprio'][None], device='cuda')
        actions = torch.as_tensor(row['actions'][0:1], device='cuda')
        valid = torch.as_tensor(row['action_valid'][0:1], device='cuda')
        def score(candidate):
            return model.expected_value(prefix, pad, robot, proprio, candidate, valid)
        proposed, telemetry = bounded_q_ascent(
            score, actions, max_normalized_rms=TRUST_RADIUS_STD,
            action_std=model.action_std)
        with torch.no_grad():
            after = score(proposed)
        records.append({'root': roots.entries[i]['candidate_group_id'],
                        'q_before': float(telemetry['q_before'][0]),
                        'q_after': float(after[0]),
                        'normalized_update_rms': float(telemetry['normalized_update_rms'][0])})
    for record in records:
        print(record)
    print('NO SIMULATOR OUTCOMES FOR CORRECTED ACTIONS; this is a gradient smoke test only.')
else:
    print('No Q-gradient diagnostic. Set RUN_DIAGNOSTIC=True to run on held-out roots.')
